16d. Ablation hybrid, Pareto hybrid nhẹ, bảng tham số và attention của GAT trên NF-ToN-IoT-v3, CPU (task 5.3, 5.8, GĐ G)

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

ALLOW_GPU = False
WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    """Đường dẫn đầu tiên khớp glob đệ quy dưới /kaggle/input (báo lỗi rõ nếu không có)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"])
assert ALLOW_GPU or INFO["device"] == "CPU", "Chưa được phép dùng GPU: tắt GPU trong cài đặt notebook"

## pytest toàn bộ (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## Đường dẫn đầu vào

In [ ]:
DS = "nf_ton_iot_v3"
DATA = os.path.dirname(find(f"{DS}/split.json"))                         # output của notebook 04
FOLDS = os.path.dirname(os.path.dirname(find(f"models_hybrid/{DS}_s42/gat_fold0.pt")))   # output của 15f
CKPT = os.path.dirname(find(f"ckpt/{DS}_gat_s42.pt"))                    # output của 07 (GAT toàn train, GCN, GraphSAGE)
INFO.update(dataset=DS, data=DATA, folds=FOLDS, ckpt=CKPT)
save_info()
print(DATA, FOLDS, CKPT, sep="\n")

## 1. Bảng tham số theo lớp và trọng số attention của GAT (task 5.3), chạy trước vì nhanh

In [ ]:
with step("gat_explain"):
    run([sys.executable, "-I", "scripts/gat_explain.py", DATA, DS, WORK, f"ckpt={CKPT}"], cwd=REPO)

## 2. Ablation hybrid (c, d), đường Pareto hybrid nhẹ và ngưỡng của hybrid đầy đủ

In [ ]:
with step("hybrid_ablation_pareto"):
    run([sys.executable, "-I", "scripts/run_hybrid_ablation.py", DATA, DS, WORK, f"folds={FOLDS}", f"full={CKPT}"], cwd=REPO)

## Kiểm tra và run_info.json

In [ ]:
import pandas as pd
chk = {}
cd = pd.read_csv(f"{WORK}/ablation_cd_{DS}.csv")
chk["ablation_cd"] = {"rows": len(cd), "models": sorted(cd["model"].unique()), "seeds": sorted(cd["seed"].unique().tolist()),
                      "only_val": set(cd["split"]) == {"val"}}
par = pd.read_csv(f"{WORK}/pareto_raw_{DS}.csv")
chk["pareto"] = {"rows": len(par), "variants": sorted(par["variant"].unique()), "seeds": sorted(par["seed"].unique().tolist())}
chk["ref_check"] = json.load(open(f"{WORK}/ref_check_{DS}.json", encoding="utf-8"))
INFO["checks"] = chk
INFO["models_light_files"] = sum(len(fs) for _, _, fs in os.walk(f"{WORK}/models_light"))
save_info()
print(json.dumps(chk, ensure_ascii=False, indent=1))
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())